In [98]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
gdrive_path = trec_root / "data/gdrive/data"
official_path = trec_root / "data/official"
output_path = trec_root / "results/rerank"

spark = get_spark()

pagerank_root = (
    dataset_root / "centrality/v2/bge-m3-knn-k15/pagerank.parquet"
).as_posix()
display(pagerank_root)
pagerank = spark.read.parquet(pagerank_root)
pagerank.show(n=5)

'/storage/home/hcoda1/8/amiyaguchi3/scratch/trec-tot-2025/data/enwiki/processed/centrality/v2/bge-m3-knn-k15/pagerank.parquet'

+---+--------------------+
| id|            pagerank|
+---+--------------------+
| 12|3.640833311601611E-8|
| 39|1.678562822591681...|
|290|4.540637012629417E-8|
|303|6.640472282136741E-8|
|305|7.482553939188427E-8|
+---+--------------------+
only showing top 5 rows


In [99]:
! head {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run

2001	Q0	1752781	0	5.0	gemini-25_alias
2001	Q0	58865	1	4.0	gemini-25_alias
2001	Q0	4907816	2	4.0	gemini-25_alias
2001	Q0	49363330	3	4.0	gemini-25_alias
2001	Q0	73331355	4	4.0	gemini-25_alias
2001	Q0	37845530	5	4.0	gemini-25_alias
2001	Q0	43936047	6	4.0	gemini-25_alias
2001	Q0	64323316	7	4.0	gemini-25_alias
2001	Q0	1650010	8	4.0	gemini-25_alias
2001	Q0	63153672	9	3.0	gemini-25_alias


In [100]:
! head {official_path}/dev3-2025-queries.jsonl

{"query_id":"2001","query":"I remember this old building I used to pass by in the heart of a bustling financial district, a place where the air always seemed thick with the scent of ambition and old money. The building itself was quite striking, with a facade that looked like it was carved out of some kind of pale stone, maybe marble? It had this grand, almost imposing presence, like it was watching over the street with a stern, unyielding gaze.  I think it was only a few stories tall, but it had this grand entrance that made it feel much larger than it actually was. The first floor was particularly memorable, almost like it was designed to be the most impressive part of the building. There were these large windows that seemed to peer out over the street, giving it a sense of openness despite its age.  I remember hearing stories about some kind of explosion or attack that happened there a long time ago, and they never really fixed the damage. Instead, they left it as a sort of testamen

In [101]:
cols = ["qid", "Q0", "docid", "rank", "score", "run_name"]
run = spark.read.csv(
    f"{gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run",
    sep="\t",
    header=False,
    inferSchema=True,
).toDF(*cols)
run.show(n=5)
run.printSchema()

+----+---+--------+----+-----+---------------+
| qid| Q0|   docid|rank|score|       run_name|
+----+---+--------+----+-----+---------------+
|2001| Q0| 1752781|   0|  5.0|gemini-25_alias|
|2001| Q0|   58865|   1|  4.0|gemini-25_alias|
|2001| Q0| 4907816|   2|  4.0|gemini-25_alias|
|2001| Q0|49363330|   3|  4.0|gemini-25_alias|
|2001| Q0|73331355|   4|  4.0|gemini-25_alias|
+----+---+--------+----+-----+---------------+
only showing top 5 rows
root
 |-- qid: integer (nullable = true)
 |-- Q0: string (nullable = true)
 |-- docid: integer (nullable = true)
 |-- rank: integer (nullable = true)
 |-- score: double (nullable = true)
 |-- run_name: string (nullable = true)



In [102]:
reranked = (
    run.join(pagerank, run.docid == pagerank.id, "inner")
    .select(
        "qid",
        "Q0",
        "docid",
        F.row_number()
        .over(Window.partitionBy("qid").orderBy(F.desc("pagerank"), "rank"))
        .alias("rank"),
        F.col("pagerank").alias("score"),
        F.col("run_name"),
    )
    .orderBy("qid", "rank")
)
reranked.show()
reranked_path = f"{output_path}/gemini-2.5-flash-dev3-pagerank-knn-k15-v1/results.txt"
Path(reranked_path).parent.mkdir(parents=True, exist_ok=True)
reranked.toPandas().to_csv(
    reranked_path,
    sep=" ",
    index=False,
    header=False,
)
! head {reranked_path}

+----+---+--------+----+--------------------+---------------+
| qid| Q0|   docid|rank|               score|       run_name|
+----+---+--------+----+--------------------+---------------+
|2001| Q0|   58865|   1|3.460409551008376E-7|gemini-25_alias|
|2001| Q0|64323316|   2|2.810500846807695E-7|gemini-25_alias|
|2001| Q0| 4907816|   3|1.782170181772658...|gemini-25_alias|
|2001| Q0|53500318|   4|1.769715486307626...|gemini-25_alias|
|2001| Q0|73331355|   5|1.692206254519868E-7|gemini-25_alias|
|2001| Q0| 7561924|   6|1.607704621952588...|gemini-25_alias|
|2001| Q0|49363330|   7|1.254840890188938...|gemini-25_alias|
|2001| Q0|37845530|   8|1.157664226791450...|gemini-25_alias|
|2001| Q0|43936047|   9|1.020356943272497...|gemini-25_alias|
|2001| Q0| 1650010|  10|9.911410547248533E-8|gemini-25_alias|
|2001| Q0| 1752781|  11|8.458954302141984E-8|gemini-25_alias|
|2001| Q0|63153672|  12|6.953395724866094E-8|gemini-25_alias|
|2002| Q0| 7946607|   1|3.019870221511079E-7|gemini-25_alias|
|2002| Q

In [103]:
! wc {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run
! wc {reranked_path}
! ls {official_path}/

  8198  49188 320004 /storage/home/hcoda1/8/amiyaguchi3/scratch/trec-tot-2025/data/gdrive/data/shared_retrieval_results/gemini-2.5-flash/dev3.run
  8114  48684 466252 /storage/home/hcoda1/8/amiyaguchi3/scratch/trec-tot-2025/results/rerank/gemini-2.5-flash-dev3-pagerank-knn-k15-v1/results.txt
dev1-2025-qrel.txt	 dev3-2025-qrel.txt	  train-2025-queries.jsonl
dev1-2025-queries.jsonl  dev3-2025-queries.jsonl  trec-tot-2025-corpus.jsonl
dev2-2025-qrel.txt	 test-2025-queries.jsonl  trec-tot-2025-offsets.jsonl
dev2-2025-queries.jsonl  train-2025-qrel.txt


In [104]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    {official_path}/dev3-2025-qrel.txt \
    {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run

recip_rank            	all	0.4001
recall_1000           	all	0.5871
ndcg_cut_10           	all	0.4357
ndcg_cut_1000         	all	0.4436


In [105]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    {official_path}/dev3-2025-qrel.txt \
    {reranked_path} 

recip_rank            	all	0.1523
recall_1000           	all	0.5871
ndcg_cut_10           	all	0.2013
ndcg_cut_1000         	all	0.2457
